# Advanced Artificial Intelligence Task 1:

In [1]:
import torch
import random
import math
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import sys
sys.path.append("..")
sys.path.append(".")
import experiment_configs.task_1_config as configs
from models.ncf import NCF
from models.lstm import LSTM
from models.sasrec import SASRec

In [ ]:
RUN_ALL=""
DATASET_PATH=""
VARIABLE_SEED=43

In [ ]:
if RUN_ALL == "NCF_Base":
    EXP = configs.ncf_baseline
elif RUN_ALL == "NCF_Deep_Layer":
    EXP = configs.ncf_deep
elif RUN_ALL == "LSTM_Base":
    EXP = configs.lstm_baseline
elif RUN_ALL == "LSTM_Seq20":
    EXP = configs.lstm_long_context
elif RUN_ALL == "SASRec_Base":
    EXP = configs.sasrec_baseline
elif RUN_ALL == "SASREC_Deep_Attn":
    EXP = configs.sasrec_heavy
else:
    EXP=configs.ncf_baseline # default fallback model

In [3]:
#EXP=ncf_baseline
#EXP=lstm_baseline
#EXP=sasrec_baseline

In [ ]:
df=pd.read_csv("data/insta_clean_data.csv")
#df=df.sample(n=100000,random_state=42)

df=df.sort_values(["user_id","order_id","cart_position"])

df=df.dropna(subset=["user_id","product_name"])

In [5]:
user_encoder=LabelEncoder()
item_encoder=LabelEncoder()

df["user_id"]=user_encoder.fit_transform(df["user_id"])
df["item_id"]=item_encoder.fit_transform(df["product_name"])+1
df["day_of_week"]=df["day_of_week"]+1
df["hour_of_day"]=df["hour_of_day"]+1

num_users=df["user_id"].nunique()
num_items=df["item_id"].nunique()

print(f"Users: {num_users} | Items: {num_items}")
print(f"Day of week range: {df["day_of_week"].min()} to {df["day_of_week"].max()}")
print(f"Hour of day range: {df["hour_of_day"].min()} to {df["hour_of_day"].max()}")

Users: 62840 | Items: 16899
Day of week range: 1 to 7
Hour of day range: 1 to 24


In [6]:
#user_sequences=df.groupby("user_id")["item_id"].apply(list)
user_sequences=df.groupby("user_id").agg({
    "item_id":list,
    "day_of_week":list,
    "hour_of_day":list
}).reset_index()

#user_sequences=[seq for seq in user_sequences if len(seq)>2]
user_sequences=user_sequences[user_sequences["item_id"].map(len)>2]

In [7]:
train_sequences=[]
test_sequences=[]

for _,row in user_sequences.iterrows():
    items=row["item_id"]
    dows=row["day_of_week"]
    hours=row["hour_of_day"]

    split_idx=int(len(items)*0.8)

    train_sequences.append({
        "items":items[:-1],
        "dows":dows[:-1],
        "hours":hours[:-1]
    })

    test_sequences.append({
        "items":items,
        "dows":dows,
        "hours":hours
    })  
print(f"Total Users: {len(user_sequences)} | Samples: {len(train_sequences)}")

Total Users: 8532 | Samples: 8532


In [8]:
class NCFDataset(Dataset):
    def __init__(self,df,num_items,num_negatives=1):
        self.users=df["user_id"].values
        self.items=df["item_id"].values
        self.dows=df["day_of_week"].values
        self.hours=df["hour_of_day"].values
        self.num_items=num_items
        self.num_negatives=num_negatives
        
        self.data=[]
        self.all_items=set(range(num_items))
        for u,i,d,h in zip(self.users,self.items,self.dows,self.hours):
            self.data.append((u,i,d,h,1))
            for _ in range(num_negatives):
                neg_i=random.randint(1,num_items)
                while neg_i == i:
                    neg_i=random.randint(1,num_items)
                self.data.append((u,neg_i,d,h,0))

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self,idx):
        user,item,dow,hour,label=self.data[idx]
        return(
            torch.tensor(user,dtype=torch.long),
            torch.tensor(item,dtype=torch.long),
            torch.tensor(dow, dtype=torch.long),
            torch.tensor(hour,dtype=torch.long),
            torch.tensor(label,dtype=torch.float)
        )

class SequenceDataset(Dataset):
    def __init__(self,sequences,max_len=10):
        self.data=[]
        self.max_len=max_len
        for seq in sequences:
            items,dows,hours=seq["items"],seq["dows"],seq["hours"]
            for i in range(1,len(items)):
                self.data.append({
                    "item_seq":items[max(0,i-max_len):i],
                    "dow_seq":dows[max(0,i-max_len):i],
                    "hour_seq":hours[max(0,i-max_len):i],
                    "target":items[i]
                })

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self,idx):
        d=self.data[idx]
        pad_len=self.max_len-len(d["item_seq"])
        items=[0]*pad_len+d["item_seq"]
        dows=[0]*pad_len+d["dow_seq"]
        hours=[0]*pad_len+d["hour_seq"]
        return(
            torch.tensor(items, dtype=torch.long),
            torch.tensor(dows,dtype=torch.long),
            torch.tensor(hours,dtype=torch.long),
            torch.tensor(d["target"],dtype=torch.long)
        )

In [9]:
if EXP.architecture=="ncf":
    dataset=NCFDataset(df,num_items=num_items,num_negatives=2)
elif EXP.architecture in ["lstm","sasrec"]:
    dataset=SequenceDataset(train_sequences,max_len=EXP.model.max_seq_len)

loader=DataLoader(dataset,batch_size=EXP.training.batch_size,shuffle=True)

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

if EXP.architecture=="ncf":
    model=NCF(num_users,num_items,EXP.model.embedding_dim)
elif EXP.architecture=="lstm":
    model=LSTM(
        num_items,
        embedding_dim=EXP.model.embedding_dim,
        hidden_dim=EXP.model.hidden_dim
    )
elif EXP.architecture=="sasrec":
    model=SASRec(
        num_items,
        embedding_dim=EXP.model.embedding_dim,
        num_heads=EXP.model.num_heads,
        num_layers=EXP.model.num_layers,
        max_len=EXP.model.max_seq_len
    )
model=model.to(device)

In [11]:
optimizer=torch.optim.Adam(model.parameters(),lr=EXP.training.learning_rate)

if EXP.architecture=="ncf":
    loss_fn=torch.nn.BCEWithLogitsLoss()
else:
    loss_fn=torch.nn.CrossEntropyLoss()

for epoch in range(EXP.training.max_epochs):
    model.train()
    total_loss=0

    for batch in loader:
        optimizer.zero_grad()

        if EXP.architecture == "ncf":
            user,item,dow,hour,label=batch
            user,item,dow,hour,label=user.to(device),item.to(device),dow.to(device),hour.to(device),label.to(device)

            pred=model(user,item,dow,hour).squeeze()
            loss=loss_fn(pred,label)

        else:
            seq,dow,hour,target=batch
            seq,dow,hour,target=seq.to(device),dow.to(device),hour.to(device),target.to(device)
            
            if (target < 1).any() or (target > num_items).any():
                print(f"INVALID TARGET: Min {target.min().item()}, Max {target.max().item()}")
                print(f"Allowed range: 1 to {num_items}")
                print()
                break

            pred=model(seq,dow,hour)
            if torch.isnan(pred).any():
                print("NaN in predictions")
                break
            loss=loss_fn(pred,target)
            #print("pred check:", torch.isnan(pred).any().item())
            # print("target check:", torch.isnan(target).any().item())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)
        optimizer.step()
        total_loss+=loss.item()
    print(f"Epoch {epoch}: Loss {total_loss/len(loader):.4f}")

KeyboardInterrupt: 

In [ ]:
def get_top_k(scores,k):
    return scores.topk(k).indices.squeeze().tolist()

def get_metrics(hit,total,precision,ndcg,mrr,k):
    return {
        f"Recall@{k}": hit/total,
        f"Precision@{k}": precision/total,
        f"NDCG@{k}": ndcg/total,
        f"MRR@{k}": mrr/total
    }

def evaluate_sequential_model(model,sequences,k=5):
    model.eval()
    recall,precision,ndcg,mrr=0,0,0,0
    total=0
    max_len=EXP.model.max_seq_len

    with torch.no_grad():
        for seq in sequences:
            if len(seq["items"])<2:
                continue
            items_in=seq["items"][:-1][-max_len:]
            dows_in=seq["dows"][:-1][-max_len:]
            hours_in=seq["hours"][:-1][-max_len:]
            true_item=seq["items"][-1]

            pad_len=max_len-len(items_in)
            items_in=[0] * pad_len + items_in
            dows_in=[0] * pad_len + dows_in
            hours_in=[0] * pad_len + hours_in
            
            item_tensor = torch.tensor(items_in).unsqueeze(0).to(device)
            dow_tensor = torch.tensor(dows_in).unsqueeze(0).to(device)
            hour_tensor = torch.tensor(hours_in).unsqueeze(0).to(device)
            
            scores = model(item_tensor,dow_tensor,hour_tensor).squeeze()
            
            top_k=get_top_k(scores,k)

            if isinstance(top_k, int):
                top_k = [top_k]

            total += 1
            if true_item in top_k:
                rank = top_k.index(true_item) + 1
                recall += 1
                precision += (1 / k)
                ndcg += 1 / math.log2(rank + 1)
                mrr += 1 / rank

    return get_metrics(recall,total,precision,ndcg,mrr,k)

def evaluate_ncf_model(model, test_data, num_items, k=5, num_neg=99):
    model.eval()

    hit = precision = ndcg = mrr = 0
    total = 0

    with torch.no_grad():
        for user, true_item, dow, hour in test_data:

            # sample negatives
            items = [true_item]
            while len(items) < num_neg + 1:
                neg = random.randint(1, num_items)
                if neg != true_item:
                    items.append(neg)

            user_tensor = torch.tensor([user]).repeat(len(items)).to(device)
            item_tensor = torch.tensor(items).to(device)
            dow_tensor = torch.tensor([dow]).repeat(len(items)).to(device)
            hour_tensor = torch.tensor([hour]).repeat(len(items)).to(device)

            scores = model(user_tensor, item_tensor, dow_tensor, hour_tensor).squeeze()

            top_k_indices = torch.topk(scores, k).indices.cpu().tolist()
            top_items = [items[i] for i in top_k_indices]

            total += 1
            if true_item in top_items:
                rank = top_items.index(true_item) + 1
                hit += 1
                precision += (1 / k)
                ndcg += 1 / math.log2(rank + 1)
                mrr += 1 / rank

    return get_metrics(hit, total, precision, ndcg, mrr, k)

k=5
if EXP.architecture == "ncf":
    num_items = int(df["item_id"].nunique())
    test_samples=[]
    for seq in test_sequences:
        if len(seq["items"])>=1:
            test_samples.append((seq.get("user_id",0), seq["items"][-1], seq["dows"][-1], seq["hours"][-1]))
    results=evaluate_ncf_model(model,test_samples,num_items,k)
else:
    results=evaluate_sequential_model(model,test_sequences,k)

print("\n===== FINAL EVALUATION =====")
for metric,value in results.items():
    print(f"{metric}: {value:.4f}")


===== FINAL EVALUATION =====
Recall@5: 0.3882
Precision@5: 0.0776
NDCG@5: 0.2308
MRR@5: 0.1795
